# 01 · AE / VAE MNIST —— 复印机 vs 摇骰子复印机

**家族位置**：`07_Generative_Models` 第 1 站。前面 01~06 全是“判别”（图进→标签出），本章反过来做“生成”（潜变量进→图出）。AE 把 28×28 压成 32 维小纸条再展开；VAE 把纸条改成“骰子”（均值+方差采样），摇一摇画出训练集没见过的新数字。MNIST 子集，CPU 分钟级。

**学习目标**
1. AE：编码-解码瓶颈，重建 MSE，能还原不能创新
2. VAE：重参数技巧 `z=μ+ε·σ` 让采样可导；ELBO = 重建 + KL
3. 潜空间走格（latent traversal）：固定 30 维、扫 2 维，看数字怎么渐变
4. 随机采样：`z~N(0,I)` 解码，直接画新图

## 1. 原理：从“背下来”到“摇出来”

### 通俗理解

**一句话**：AE 是复印机——28×28 的图压成 32 个数的小纸条（瓶颈），再展开还原；纸条只能记见过的，画不出新东西。VAE 是摇骰子复印机——纸条改写成“均值+方差”，每次复印先摇骰子采样，摇出的点都能解码成图，所以能画新数字。

**比喻**：AE 的潜空间像仓库货架——每个 Bahia 位只放一张见过的图；VAE 的 KL 项像交通规则，逼所有货架排成标准正态小区，空位也被“合法化”，随机去小区里指一个地址就能开出新图。

### 结构账

```
编码： 1×28×28 → Conv(1→32,s2) → Conv(32→64,s2) → 64×7×7 拍扁 → 32 维
AE：    z = Linear(h)；x̂ = 反卷积(z)；Loss = MSE(x̂, x)
VAE：   μ,logσ² = Linear(h)；z = μ + ε·exp(0.5·logσ²)；Loss = BCE + KL
解码：  Linear(32→64×7×7) → TConv(64→32,s2) → TConv(32→1,s2) → 28×28
训练：  MNIST 子集 train 6000/val 1000，Adam 1e-3，batch 128，AE 8ep / VAE 10ep
```

- **评估**：test 重建 MSE/BCE + 重建拼图 + 潜空间走格 + 随机采样

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import load_mnist_local
from common.models import ConvAE, ConvVAE
from common.engine import fit_ae, fit_vae
from common.utils import set_seed, setup_chinese_font, count_params, show_mnist

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
print("torch:", torch.__version__)

N_TRAIN, N_VAL = 6000, 1000
EPOCHS_AE, EPOCHS_VAE, BATCH, LR = 8, 10, 128, 1e-3
Xtr, ytr, Xva, yva, Xte, yte = load_mnist_local(N_TRAIN, N_VAL, seed=0)
print(f"train {tuple(Xtr.shape)} / val {tuple(Xva.shape)} / test {tuple(Xte.shape)}")


## 2. AE：复印机（能还原，不能创新）

In [ ]:
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=BATCH, shuffle=True)
val_loader = DataLoader(TensorDataset(Xva, yva), batch_size=512)

ae = ConvAE(latent_dim=32)
print(f"AE params={count_params(ae)}")
torch.manual_seed(0)
hist_ae = fit_ae(ae, train_loader, val_loader, epochs=EPOCHS_AE, lr=LR)

# fig1：AE 收敛曲线
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(hist_ae["train"], label="train", color="#4C72B0", marker="o")
ax.plot(hist_ae["val"], label="val", color="#DD8452", marker="s")
ax.set_xlabel("epoch"); ax.set_ylabel("MSE (sum/图)")
ax.set_title("AE 重建收敛（瓶颈 32 维）")
ax.legend()
plt.tight_layout()
plt.savefig(FIGS / "fig1_ae_loss.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. VAE：摇骰子复印机（重参数 + KL 拉回正态）

In [ ]:
vae = ConvVAE(latent_dim=32)
print(f"VAE params={count_params(vae)}")
torch.manual_seed(0)
hist_vae = fit_vae(vae, train_loader, val_loader, epochs=EPOCHS_VAE, lr=LR, beta=1.0)

# fig2：VAE 曲线（ELBO + BCE/KL 拆分）
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(hist_vae["train"], label="train ELBO", color="#4C72B0", marker="o")
axes[0].plot(hist_vae["val"], label="val ELBO", color="#DD8452", marker="s")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("ELBO")
axes[0].set_title("VAE 收敛")
axes[0].legend()
axes[1].plot(hist_vae["bce"], label="BCE 重建", color="#4C72B0")
axes[1].plot(hist_vae["kl"], label="KL 正则", color="#55A868")
axes[1].set_xlabel("epoch"); axes[1].legend()
axes[1].set_title("BCE vs KL（KL 被压住=潜空间贴住正态）")
plt.tight_layout()
plt.savefig(FIGS / "fig2_vae_loss.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. 对比：重建拼图（原图 vs AE vs VAE）

In [ ]:
import torch.nn.functional as F
# test 重建误差（同口径 MSE/图）
ae.eval(); vae.eval()
with torch.no_grad():
    xb = Xte[:1000]
    mse_ae = F.mse_loss(ae(xb), xb, reduction="sum").item() / len(xb)
    mu, logvar = vae.encode(xb)
    z = vae.reparam(mu, logvar)
    mse_vae = F.mse_loss(torch.sigmoid(vae.decode(z)), (xb * 0.3081 + 0.1307).clamp(0, 1), reduction="sum").item() / len(xb)
print(f"test 重建 MSE/图：AE={mse_ae:.1f} VAE={mse_vae:.1f}（口径：AE 在标准化域比，VAE 在 [0,1] 域比——见 README §8）")

# fig3：8 张三行拼图
with torch.no_grad():
    xs = Xte[:8]
    ra = ae(xs)
    mu8, lv8 = vae.encode(xs)
    rv = torch.sigmoid(vae.decode(vae.reparam(mu8, lv8)))
fig, axes = plt.subplots(3, 8, figsize=(10, 4.0))
for j in range(8):
    axes[0][j].imshow(show_mnist(xs[j]), cmap="gray"); axes[0][j].axis("off")
    axes[1][j].imshow(show_mnist(ra[j]), cmap="gray"); axes[1][j].axis("off")
    axes[2][j].imshow(show_mnist(rv[j]), cmap="gray"); axes[2][j].axis("off")
axes[0][0].set_ylabel("原图", fontsize=10); axes[1][0].set_ylabel("AE", fontsize=10); axes[2][0].set_ylabel("VAE", fontsize=10)
plt.suptitle("重建对比：AE 更锐（确定性），VAE 偏糊（采样噪声+KL 约束）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig3_recon.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. 生成：潜空间走格 + 随机采样（VAE 的独门绝技，AE 做不到）

In [ ]:
# fig4：走格——取测试均值潜向量为中心，扫 z[0],z[1] 各 ±2σ，看数字渐变
vae.eval()
with torch.no_grad():
    mu_all, _ = vae.encode(Xte[:2000])
    center = mu_all.mean(dim=0)  # (32,)
    grid = 8
    zs = center.repeat(grid*grid, 1)
    lin = torch.linspace(-2.5, 2.5, grid)
    for i in range(grid):
        for j in range(grid):
            zs[i*grid+j, 0] = lin[j]
            zs[i*grid+j, 1] = lin[i]
    imgs = torch.sigmoid(vae.decode(zs))
fig, axes = plt.subplots(grid, grid, figsize=(7, 7))
for i in range(grid):
    for j in range(grid):
        axes[i][j].imshow(show_mnist(imgs[i*grid+j]), cmap="gray")
        axes[i][j].axis("off")
plt.suptitle("VAE 潜空间走格（z0→横，z1→纵）：相邻格渐变=空间连续", fontsize=12)
plt.tight_layout()
plt.savefig(FIGS / "fig4_traverse.png", dpi=150, bbox_inches="tight")
plt.show()

# fig5：随机采样 16 张（z~N(0,I)，训练集没见过的全新数字）
with torch.no_grad():
    samp = vae.sample(16, device="cpu")
fig, axes = plt.subplots(2, 8, figsize=(10, 2.8))
for ax, k in zip(axes.flat, range(16)):
    ax.imshow(show_mnist(samp[k]), cmap="gray")
    ax.axis("off")
plt.suptitle("VAE 随机采样（z~N(0,I) 解码：全新数字，非复制）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig5_sample.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"AE test MSE={mse_ae:.1f} | VAE test MSE={mse_vae:.1f}")


## 6. 总结与下一步

**本项目收获**

1. AE 复印机闭环：32 维瓶颈，test 重建 MSE 见 fig3（确定性=更锐）
2. VAE 摇骰子：重参数让采样可导，KL 把潜空间压成正态小区
3. 走格渐变 + 随机采样：AE 做不到的两件事（仓库货架 vs 连续小区）
4. 代价：VAE 重建偏糊（采样噪声+KL 约束），想要又锐又能采样→看 GAN/Diffusion

**下一步**：`02_GAN_MNIST_CIFAR10`（印假钞 vs 验钞机对抗，MLP-GAN vs DCGAN，模式坍塌实拍）→ `03_Diffusion_MNIST`（雾玻璃擦雾，T 步去噪链）。